In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path


os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [22]:
df_oms = pd.DataFrame(columns=['country_name','city_orig','city_code_orig','station_name','station_id_orig','type_of_station_orig',
                          'type_of_station_detailed','latitude','longitude','adress','year',
                           'pm10_concentration','pm10_tempcov','pm10_tempcov_days','equipment_pm10','pop_cov_pm10',
                           'pm25_concentration','pm25_tempcov','pm25_tempcov_days','equipment_pm25','pop_cov_pm25',
                           'no2_concentration','no2_tempcov','no2_tempcov_days','equipment_no2','pop_cov_no2',
                           'source_web_link','other_pollutants','population','population_source','pop_year','comments'])

pastas = ['MP10','MP25','NO2']

pop_buffer = pd.read_csv(os.getcwd()+'/data/outputs/populacao_varbuf.csv')

pop_mun = pd.read_csv(os.getcwd()+'/data/dicionarios/br_ibge_populacao_municipio.csv')
pop_mun = pop_mun[pop_mun['ano']==2022]

type_station = pd.read_csv(os.getcwd()+'/data/outputs/uso_solo_varbuf.csv')

monitoramento_qar = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')
monitoramento_pols = monitoramento_qar[monitoramento_qar['POLUENTE'].isin(['MP10','MP25','NO2'])]

list_id_mma = monitoramento_pols['ID_MMA'].unique()

for id_mma in list_id_mma:

    dados_estacao = monitoramento_pols.loc[monitoramento_pols['ID_MMA'] == id_mma, 
            ['CIDADE','CD_MUN', 'ID_OEMA', 'ID_MMA', 'LATITUDE', 'LONGITUDE']].reset_index()

    try:
        tipo_solo = type_station[type_station['ID_MMA']==id_mma]['GRUPO_PRED_VAR'].reset_index()['GRUPO_PRED_VAR'][0]
        populacao = pop_mun[pop_mun['id_municipio']==dados_estacao['CD_MUN'][0]]['populacao'].reset_index()['populacao'][0]
    except:
        tipo_solo = 'Não há dado de localização da estação'
        populacao = 'Não há dados de população'
    
    linha_geral = {'country_name':['Brasil'],
                 'city_orig':[dados_estacao['CIDADE'][0]],
                 'city_code_orig':[dados_estacao['CD_MUN'][0]],
                 'station_name':[dados_estacao['ID_OEMA'][0]],
                 'station_id_orig':[id_mma],
                 'type_of_station_orig':[tipo_solo],
                 'type_of_station_detailed':[''],
                 'latitude':[dados_estacao['LATITUDE'][0]],
                 'longitude':[dados_estacao['LONGITUDE'][0]],
                 'year':[''],
                 'pm10_concentration':[''],
                 'pm10_tempcov':[''],
                 'equipment_pm10':[''],
                 'pop_cov_pm10':[''],     
                 'pm25_concentration':[''],
                 'pm25_tempcov':[''],
                 'equipment_pm25':[''],
                 'pop_cov_pm25':[''],      
                 'no2_concentration':[''],
                 'no2_tempcov':[''],
                 'equipment_no2':[''],
                 'pop_cov_no2':[''],
                 'source_web_link':[''],
                 'other_pollutants':[''],
                 'population':[populacao],
                 'population_source':['IBGE'],
                 'pop_year':[2025],
                 'comments':['']}

    for ano in range(2010,2025):

        valor = 0

        linha_geral_ano = linha_geral

        linha_geral_ano['year'] = [ano]
            
        for pol in pastas:
    
            if pol == 'MP10':
                variavel = 'pm10'
            elif pol == 'MP25':
                variavel = 'pm25'
            elif pol == 'NO2':
                variavel = 'no2'
    
            monitoramento_pol = monitoramento_pols[monitoramento_pols['POLUENTE']==pol]
            monitoramento_estacao = monitoramento_pol[monitoramento_pol['ID_MMA']==id_mma].reset_index() 

            if len(monitoramento_estacao) > 0:
                
                id_mma_completo = monitoramento_estacao['ID_MMA_COMPLETO'][0]

                try:
                    pop_cov = pop_buffer[pop_buffer['ID_MMA_COMPLETO']==id_mma_completo]['POP_BUFFER'].reset_index()['POP_BUFFER'][0]
                except:
                    pop_cov = 'Sem dados de cobertura da estação'
                
                try:
                    equipment = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO'].reset_index()['MODELO'][0]
                except:
                    equipment = 'Sem dados de equipamento'
                    
                rep_espacial = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['REP_ESPACIAL'].reset_index()['REP_ESPACIAL'][0]
                
                if (id_mma_completo+'.csv') in os.listdir(os.getcwd()+'/data/MQAr_averages/anual/'+pol):

                    df = pd.read_csv(os.getcwd()+'/data/MQAr_averages/anual/'+pol+'/'+id_mma_completo+'.csv')

                    df = df[df['ANO']==ano].reset_index()

                    print('Entrou no if: '+id_mma_completo)

                    if len(df)>0:
                        
                        print(ano)

                        valor = valor + 1
                        
                        tempcov = df['PRCNT_DIAS_ANO_REP_TEMPORAL'][0]
                        concentration = df['VALOR'][0]

                        linha_geral_ano[variavel+'_concentration'] = [concentration]
                        linha_geral_ano[variavel+'_tempcov'] = [tempcov]
                        linha_geral_ano['equipment_'+variavel] = [equipment]
                        linha_geral_ano['pop_cov_'+variavel] = [int(pop_cov)]

        if valor > 0:
            
            df_linha = pd.DataFrame(linha_geral_ano)
            
            df_oms = pd.concat([df_oms, df_linha], ignore_index=True)
            

Entrou no if: BA0010ND001
2010
Entrou no if: BA0010ND004
2010
Entrou no if: BA0010ND001
2011
Entrou no if: BA0010ND004
2011
Entrou no if: BA0010ND001
2012
Entrou no if: BA0010ND004
2012
Entrou no if: BA0010ND001
2013
Entrou no if: BA0010ND004
2013
Entrou no if: BA0010ND001
2014
Entrou no if: BA0010ND004
2014
Entrou no if: BA0010ND001
2015
Entrou no if: BA0010ND004
2015
Entrou no if: BA0010ND001
2016
Entrou no if: BA0010ND004
2016
Entrou no if: BA0010ND001
2017
Entrou no if: BA0010ND004
2017
Entrou no if: BA0010ND001
2018
Entrou no if: BA0010ND004
2018
Entrou no if: BA0010ND001
2019
Entrou no if: BA0010ND004
2019
Entrou no if: BA0010ND001
2020
Entrou no if: BA0010ND004
2020
Entrou no if: BA0010ND001
2021
Entrou no if: BA0010ND004
2021
Entrou no if: BA0010ND001
2022
Entrou no if: BA0010ND004
2022


/tmp/ipykernel_643186/4260606010.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_oms = pd.concat([df_oms, df_linha], ignore_index=True)


Entrou no if: BA0010ND001
2023
Entrou no if: BA0010ND004
2023
Entrou no if: BA0010ND001
2024
Entrou no if: BA0010ND004
2024
Entrou no if: BA0013ND001
2010
Entrou no if: BA0013ND004
2010
Entrou no if: BA0013ND001
2011
Entrou no if: BA0013ND004
2011
Entrou no if: BA0013ND001
2012
Entrou no if: BA0013ND004
2012
Entrou no if: BA0013ND001
2013
Entrou no if: BA0013ND004
2013
Entrou no if: BA0013ND001
2014
Entrou no if: BA0013ND004
2014
Entrou no if: BA0013ND001
2015
Entrou no if: BA0013ND004
2015
Entrou no if: BA0013ND001
2016
Entrou no if: BA0013ND004
2016
Entrou no if: BA0013ND001
2017
Entrou no if: BA0013ND004
2017
Entrou no if: BA0013ND001
2018
Entrou no if: BA0013ND004
2018
Entrou no if: BA0013ND001
2019
Entrou no if: BA0013ND004
2019
Entrou no if: BA0013ND001
2020
Entrou no if: BA0013ND004
2020
Entrou no if: BA0013ND001
2021
Entrou no if: BA0013ND004
2021
Entrou no if: BA0013ND001
2022
Entrou no if: BA0013ND004
2022
Entrou no if: BA0013ND001
2023
Entrou no if: BA0013ND004
2023
Entrou n

KeyboardInterrupt: 

In [23]:
df_oms.columns

Index(['country_name', 'city_orig', 'city_code_orig', 'station_name',
       'station_id_orig', 'type_of_station_orig', 'type_of_station_detailed',
       'latitude', 'longitude', 'adress', 'year', 'pm10_concentration',
       'pm10_tempcov', 'pm10_tempcov_days', 'equipment_pm10', 'pop_cov_pm10',
       'pm25_concentration', 'pm25_tempcov', 'pm25_tempcov_days',
       'equipment_pm25', 'pop_cov_pm25', 'no2_concentration', 'no2_tempcov',
       'no2_tempcov_days', 'equipment_no2', 'pop_cov_no2', 'source_web_link',
       'other_pollutants', 'population', 'population_source', 'pop_year',
       'comments'],
      dtype='object')

In [1]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
from datetime import datetime, timedelta
import re
import numpy as np
from pathlib import Path


os.chdir('/home/nobre/Notebooks/RQAR_2025_book/')

In [2]:
df_oms = pd.DataFrame(columns=['country_name','city_orig','city_code_orig','station_name','station_id_orig','type_of_station_orig',
                          'type_of_station_detailed','latitude','longitude','adress','year',
                           'pm10_concentration','pm10_tempcov','pm10_tempcov_days','equipment_pm10','pop_cov_pm10',
                           'pm25_concentration','pm25_tempcov','pm25_tempcov_days','equipment_pm25','pop_cov_pm25',
                           'no2_concentration','no2_tempcov','no2_tempcov_days','equipment_no2','pop_cov_no2',
                           'source_web_link','other_pollutants','population','population_source','pop_year','comments'])

pastas = ['MP10','MP25','NO2']

pop_buffer = pd.read_csv(os.getcwd()+'/data/outputs/populacao_varbuf.csv')

pop_mun = pd.read_csv(os.getcwd()+'/data/dicionarios/br_ibge_populacao_municipio.csv')
pop_mun = pop_mun[pop_mun['ano']==2022]

type_station = pd.read_csv(os.getcwd()+'/data/outputs/uso_solo_varbuf.csv')

monitoramento_qar = pd.read_csv(os.getcwd()+'/data/Monitoramento_QAr_BR.csv')
monitoramento_pols = monitoramento_qar[monitoramento_qar['POLUENTE'].isin(['MP10','MP25','NO2'])]

list_id_mma = monitoramento_pols['ID_MMA'].unique()

for id_mma in list_id_mma:

    dados_estacao = monitoramento_pols.loc[monitoramento_pols['ID_MMA'] == id_mma, 
            ['CIDADE','CD_MUN', 'ID_OEMA', 'ID_MMA', 'LATITUDE', 'LONGITUDE']].reset_index()

    try:
        tipo_solo = type_station[type_station['ID_MMA']==id_mma]['GRUPO_PRED_VAR'].reset_index()['GRUPO_PRED_VAR'][0]
        populacao = pop_mun[pop_mun['id_municipio']==dados_estacao['CD_MUN'][0]]['populacao'].reset_index()['populacao'][0]
    except:
        tipo_solo = 'Não há dado de localização da estação'
        populacao = 'Não há dados de população'
    
    linha_geral = {'country_name':['Brasil'],
                 'city_orig':[dados_estacao['CIDADE'][0]],
                 'city_code_orig':[dados_estacao['CD_MUN'][0]],
                 'station_name':[dados_estacao['ID_OEMA'][0]],
                 'station_id_orig':[id_mma],
                 'type_of_station_orig':[tipo_solo],
                 'type_of_station_detailed':[''],
                 'latitude':[dados_estacao['LATITUDE'][0]],
                 'longitude':[dados_estacao['LONGITUDE'][0]],
                 'year':[''],
                 'pm10_concentration':[''],
                 'pm10_tempcov':[''],
                 'equipment_pm10':[''],
                 'pop_cov_pm10':[''],     
                 'pm25_concentration':[''],
                 'pm25_tempcov':[''],
                 'equipment_pm25':[''],
                 'pop_cov_pm25':[''],      
                 'no2_concentration':[''],
                 'no2_tempcov':[''],
                 'equipment_no2':[''],
                 'pop_cov_no2':[''],
                 'source_web_link':[''],
                 'other_pollutants':[''],
                 'population':[populacao],
                 'population_source':['IBGE'],
                 'pop_year':[2025],
                 'comments':['']}

    for ano in range(2010,2025):

        valor = 0

        linha_geral_ano = linha_geral

        linha_geral_ano['year'] = [ano]

        dict_
            
        for pol in pastas:
    
            if pol == 'MP10':
                variavel = 'pm10'
            elif pol == 'MP25':
                variavel = 'pm25'
            elif pol == 'NO2':
                variavel = 'no2'
    
            monitoramento_pol = monitoramento_pols[monitoramento_pols['POLUENTE']==pol]
            monitoramento_estacao = monitoramento_pol[monitoramento_pol['ID_MMA']==id_mma].reset_index() 

            if len(monitoramento_estacao) > 0:
                
                id_mma_completo = monitoramento_estacao['ID_MMA_COMPLETO'][0]

                try:
                    pop_cov = pop_buffer[pop_buffer['ID_MMA_COMPLETO']==id_mma_completo]['POP_BUFFER'].reset_index()['POP_BUFFER'][0]
                except:
                    pop_cov = 'Sem dados de cobertura da estação'
                
                try:
                    equipment = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['MODELO'].reset_index()['MODELO'][0]
                except:
                    equipment = 'Sem dados de equipamento'
                    
                rep_espacial = monitoramento_estacao[monitoramento_estacao['ID_MMA_COMPLETO']==id_mma_completo]['REP_ESPACIAL'].reset_index()['REP_ESPACIAL'][0]
                
                if (id_mma_completo+'.csv') in os.listdir(os.getcwd()+'/data/MQAr_averages/anual/'+pol):

                    df = pd.read_csv(os.getcwd()+'/data/MQAr_averages/anual/'+pol+'/'+id_mma_completo+'.csv')

                    df = df[df['ANO']==ano].reset_index()

                    print('Entrou no if: '+id_mma_completo)

                    if len(df)>0:
                        
                        print(ano)

                        valor = valor + 1
                        
                        tempcov = df['PRCNT_DIAS_ANO_REP_TEMPORAL'][0]
                        concentration = df['VALOR'][0]

                        linha_geral_ano[variavel+'_concentration'] = [concentration]
                        linha_geral_ano[variavel+'_tempcov'] = [tempcov]
                        linha_geral_ano['equipment_'+variavel] = [equipment]
                        linha_geral_ano['pop_cov_'+variavel] = [int(pop_cov)]

        if valor > 0:
            
            df_linha = pd.DataFrame(linha_geral_ano)
            
            df_oms = pd.concat([df_oms, df_linha], ignore_index=True)
            

Entrou no if: BA0010ND001
2010
Entrou no if: BA0010ND004
2010
Entrou no if: BA0010ND001
2011
Entrou no if: BA0010ND004
2011
Entrou no if: BA0010ND001
2012
Entrou no if: BA0010ND004
2012
Entrou no if: BA0010ND001
2013
Entrou no if: BA0010ND004
2013
Entrou no if: BA0010ND001
2014
Entrou no if: BA0010ND004
2014
Entrou no if: BA0010ND001
2015
Entrou no if: BA0010ND004
2015
Entrou no if: BA0010ND001
2016
Entrou no if: BA0010ND004
2016
Entrou no if: BA0010ND001
2017
Entrou no if: BA0010ND004
2017
Entrou no if: BA0010ND001
2018
Entrou no if: BA0010ND004
2018
Entrou no if: BA0010ND001
2019
Entrou no if: BA0010ND004
2019
Entrou no if: BA0010ND001
2020
Entrou no if: BA0010ND004
2020
Entrou no if: BA0010ND001
2021
Entrou no if: BA0010ND004
2021
Entrou no if: BA0010ND001
2022


/tmp/ipykernel_699397/1665183121.py:125: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_oms = pd.concat([df_oms, df_linha], ignore_index=True)


Entrou no if: BA0010ND004
2022
Entrou no if: BA0010ND001
2023
Entrou no if: BA0010ND004
2023
Entrou no if: BA0010ND001
2024
Entrou no if: BA0010ND004
2024
Entrou no if: BA0013ND001
2010
Entrou no if: BA0013ND004
2010
Entrou no if: BA0013ND001
2011
Entrou no if: BA0013ND004
2011
Entrou no if: BA0013ND001
2012
Entrou no if: BA0013ND004
2012
Entrou no if: BA0013ND001
2013
Entrou no if: BA0013ND004
2013
Entrou no if: BA0013ND001
2014
Entrou no if: BA0013ND004
2014
Entrou no if: BA0013ND001
2015
Entrou no if: BA0013ND004
2015
Entrou no if: BA0013ND001
2016
Entrou no if: BA0013ND004
2016
Entrou no if: BA0013ND001
2017
Entrou no if: BA0013ND004
2017
Entrou no if: BA0013ND001
2018
Entrou no if: BA0013ND004
2018
Entrou no if: BA0013ND001
2019
Entrou no if: BA0013ND004
2019
Entrou no if: BA0013ND001
2020
Entrou no if: BA0013ND004
2020
Entrou no if: BA0013ND001
2021
Entrou no if: BA0013ND004
2021
Entrou no if: BA0013ND001
2022
Entrou no if: BA0013ND004
2022
Entrou no if: BA0013ND001
2023
Entrou n

ValueError: invalid literal for int() with base 10: 'Sem dados de cobertura da estação'

In [23]:
df_oms.columns

Index(['country_name', 'city_orig', 'city_code_orig', 'station_name',
       'station_id_orig', 'type_of_station_orig', 'type_of_station_detailed',
       'latitude', 'longitude', 'adress', 'year', 'pm10_concentration',
       'pm10_tempcov', 'pm10_tempcov_days', 'equipment_pm10', 'pop_cov_pm10',
       'pm25_concentration', 'pm25_tempcov', 'pm25_tempcov_days',
       'equipment_pm25', 'pop_cov_pm25', 'no2_concentration', 'no2_tempcov',
       'no2_tempcov_days', 'equipment_no2', 'pop_cov_no2', 'source_web_link',
       'other_pollutants', 'population', 'population_source', 'pop_year',
       'comments'],
      dtype='object')

In [5]:
df_oms.to_csv('/home/nobre/Notebooks/RQAR_2025_book/data/dicionarios/envio_oms.csv', index=False)